In [ ]:
!pip install transformers datasets torch pandas matplotlib soundfile librosa

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
import soundfile as sf
import librosa
from tqdm.auto import tqdm
import time
from transformers import pipeline


In [ ]:

# models configuration
WHISPER_MODEL = "openai/whisper-base"  # audio transcription
QA_MODEL = "deepset/roberta-base-squad2"  # qa model for question answering

# Phase 1 results
PHASE1_JSON = "librispeech_local_1500.json"



asr_pipeline = pipeline(
    "automatic-speech-recognition",
    model=WHISPER_MODEL,
    device=-1,  
    generate_kwargs={"language": "english"}
)

qa_pipeline = pipeline(
    "question-answering",
    model=QA_MODEL,
    device=-1  
)

In [ ]:
class SpeechQASystem:
    """
    process:
    1. Speech input, trasnscription, question text
    2. text input, QA system, answer
    """
    
    def __init__(self, asr_pipeline, qa_pipeline):
        self.asr = asr_pipeline
        self.qa = qa_pipeline
        self.stats = {
            'total_queries': 0,
            'successful_queries': 0,
            'failed_queries': 0,
            'avg_asr_time': 0,
            'avg_qa_time': 0,
            'avg_total_time': 0
        }
    
    def transcribe_audio(self, audio_path=None, audio_array=None, sample_rate=16000):
        """
        Args:
            audio_path
            audio_array
            sample_rate
        
        Returns:
            str: transcription text
        """
        start_time = time.time()
        
        try:
            if audio_path:
                # read audio
                audio_array, sample_rate = sf.read(audio_path)
            
            # transcribe
            result = self.asr({
                'array': audio_array,
                'sampling_rate': sample_rate
            })
            
            transcription = result['text'].strip()
            asr_time = time.time() - start_time
            
            return transcription, asr_time
        
        except Exception as e:
            print(f"error: {e}")
            return "", 0
    
    def answer_question(self, question, context):
        """
        
        Args:
            question
            context
        
        Returns:
            dict: answer and confidence
        """
        start_time = time.time()
        
        try:
            result = self.qa(
                question=question,
                context=context
            )
            
            qa_time = time.time() - start_time
            
            return {
                'answer': result['answer'],
                'confidence': result['score'],
                'qa_time': qa_time
            }
        
        except Exception as e:
            print(f"QA error: {e}")
            return {
                'answer': '',
                'confidence': 0,
                'qa_time': 0
            }
    
    def speech_qa(self, audio_path, context, show_details=True):
        """
        Args:
            audio_path
            context
            show_details
        
        Returns:
            dict: result
        """
        total_start = time.time()
        
        #transcribe audio
        question_text, asr_time = self.transcribe_audio(audio_path=audio_path)
        
        if not question_text:
            self.stats['failed_queries'] += 1
            return {
                'success': False,
                'error': 'ASR failed'
            }
        
        
        # answer question
        qa_result = self.answer_question(question_text, context)
        
        if show_details:
            print(f"answer: {qa_result['answer']}")
            print(f"confidence level: {qa_result['confidence']:.4f}")
        
        total_time = time.time() - total_start
        
        # update stats
        self.stats['total_queries'] += 1
        self.stats['successful_queries'] += 1
        self.stats['avg_asr_time'] = (
            (self.stats['avg_asr_time'] * (self.stats['successful_queries'] - 1) + asr_time) / 
            self.stats['successful_queries']
        )
        self.stats['avg_qa_time'] = (
            (self.stats['avg_qa_time'] * (self.stats['successful_queries'] - 1) + qa_result['qa_time']) / 
            self.stats['successful_queries']
        )
        self.stats['avg_total_time'] = (
            (self.stats['avg_total_time'] * (self.stats['successful_queries'] - 1) + total_time) / 
            self.stats['successful_queries']
        )
        
        return {
            'success': True,
            'question_audio': audio_path,
            'question_text': question_text,
            'answer': qa_result['answer'],
            'confidence': qa_result['confidence'],
            'timing': {
                'asr_time': asr_time,
                'qa_time': qa_result['qa_time'],
                'total_time': total_time
            }
        }
    
    def get_statistics(self):
        """获取系统统计信息"""
        return self.stats

# create system instance
speech_qa_system = SpeechQASystem(asr_pipeline, qa_pipeline)


In [ ]:
#load phase1 data
with open(PHASE1_JSON, 'r', encoding='utf-8') as f:
    phase1_data = json.load(f)
    
# extract transcriptions
transcriptions = [
        {
            'index': r['index'],
            'id': r['id'],
            'text': r['transcription']
        }
        for r in phase1_data['results'] if r['success']
    ]
print(f"  {transcriptions[0]['text']}...")
    

In [ ]:
#create test questions
test_questions_text = [
    "What does he hope?"
]

demo_results_text = []

# test on first 2 transcriptions
for i, trans in enumerate(transcriptions[:2]):
    print(f"ID: {trans['id']}")
    print(f"text: {trans['text'][:200]}...")
    print()
    
    for question in test_questions_text: 
        print(f"\nQ: {question}")
        
        result = qa_pipeline(
            question=question,
            context=trans['text']
        )
        
        print(f"A: {result['answer']}")
        print(f"confidence: {result['score']:.2f}")
        
        if result['score'] > 0.5:
            print("high confidence")
        elif result['score'] > 0.3:
            print("medium confidence")
        else:
            print("unknown")
        
        demo_results_text.append({
            'context_id': trans['id'],
            'question': question,
            'answer': result['answer'],
            'confidence': result['score']
        })


***real examples***

In [ ]:
# multiple audios
audio_files = [
    "food.wav",  
    "pizza.wav",
]

# knowledge base
context = "Italy's favorite foods include:Pizza: The iconic dish, especially the Neapolitan style.Easy, cheap, and filling, pizza has long been a common snack or meal, especially in Naples where tomato sauce was first added."

# process it
for i, audio in enumerate(audio_files):
    result = speech_qa_system.speech_qa(audio, context)
    
    if result['success']:
        print(f"  Q: {result['question_text']}")
        print(f"  A: {result['answer']}")
        print(f"  confidence: {result['confidence']:.2f}")
        if result['confidence'] > 0.5:
            print("reliable answer")
        else:
            print("answer may be not true")
    
    else:
        print(f"failed to process")